# ==========================================
# 10 Academy AI Mastery — Week 9 Challenge
# Task 2: Build Time Series Forecasting Models
# ==========================================

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Set styling for clarity
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available() else 'default')

# ==========================================
# STEP 1: Prepare Data & Chronological Split
# ==========================================
print("--- Step 1: Data Acquisition & Split ---")
# Fetch clean continuous dataset for Tesla (TSLA)
df = yf.download('TSLA', start='2015-01-01', end='2026-06-30')['Adj Close']
df = df.ffill().bfill()
df_frame = pd.DataFrame(df)

# Chronological split to preserve temporal sequence order (No random shuffling)
# Training set: Jan 2015 - Dec 2024 | Testing set: Jan 2025 - June 2026
train_series = df_frame.loc[:'2024-12-31']['Adj Close']
test_series = df_frame.loc['2025-01-01':]['Adj Close']

print(f"Training Data Pool Size : {train_series.shape[0]} days")
print(f"Testing Data Pool Size  : {test_series.shape[0]} days\n")

# ==========================================
# STEP 2: Implement & Optimize ARIMA Model
# ==========================================
print("--- Step 2: Training Optimized ARIMA Baseline ---")

# auto_arima uses Akaike Information Criterion (AIC) minimizing grid search 
# to select optimal integration boundaries (p, d, q)
arima_model = auto_arima(
    train_series, 
    seasonal=False,       # Adjust to True with 'm' if running structural SARIMA
    stepwise=True, 
    suppress_warnings=True, 
    error_action='ignore',
    trace=True
)

print("\n[Optimal ARIMA Parameters Chosen]")
print(arima_model.summary())

# Generate dynamic out-of-sample forecasts matching test window horizon length
arima_forecast = arima_model.predict(n_periods=len(test_series))
arima_forecast.index = test_series.index

# ==========================================
# STEP 3: Implement & Optimize LSTM Model
# ==========================================
print("\n--- Step 3: Structuring and Training Deep LSTM Network ---")

# Step 3a: Normalize price distributions between [0, 1] for stable gradient descent
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_train = scaler.fit_transform(train_series.values.reshape(-1, 1))
scaled_test = scaler.transform(test_series.values.reshape(-1, 1))

# Step 3b: Windowing Sequence Vectorization function
def create_sequences(data, window_size=60):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

WINDOW_SIZE = 60
X_train, y_train = create_sequences(scaled_train, window_size=WINDOW_SIZE)

# For testing data, prepend the last 60 days of training to capture boundary forecasts
combined_test_input = np.vstack((scaled_train[-WINDOW_SIZE:], scaled_test))
X_test, y_test = create_sequences(combined_test_input, window_size=WINDOW_SIZE)

# Step 3c: Define stacked LSTM architectural nodes
model = Sequential([
    LSTM(units=50, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2), # Mitigate overfitting profiles
    LSTM(units=50, return_sequences=False),
    Dropout(0.2),
    Dense(units=25),
    Dense(units=1) # Single-step numeric regression output
])

model.compile(optimizer='adam', loss='mean_squared_error')
print("\n[LSTM Neural Architecture Layout Summary]")
model.summary()

# Step 3d: Fit sequence model
print("\nTraining network weights...")
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# Step 3e: Generate deep predictions and reverse min-max normalization scaling matrix
lstm_scaled_predictions = model.predict(X_test)
lstm_forecast = scaler.inverse_transform(lstm_scaled_predictions).flatten()

# ==========================================
# STEP 4: Evaluate and Compare Models
# ==========================================
print("\n--- Step 4: Metric Performance Compilation ---")

def calculate_mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100

# Metric Arrays
metrics_summary = {}
for name, preds in [("ARIMA", arima_forecast), ("LSTM", lstm_forecast)]:
    mae = mean_absolute_error(test_series.values, preds)
    rmse = np.sqrt(mean_squared_error(test_series.values, preds))
    mape = calculate_mape(test_series.values, preds)
    metrics_summary[name] = [f"{mae:.4f}", f"{rmse:.4f}", f"{mape:.2f}%"]

# Convert performance profiles to a clear DataFrame summary table
comparison_table = pd.DataFrame(
    metrics_summary, 
    index=['MAE (Mean Absolute Error)', 'RMSE (Root Mean Squared Error)', 'MAPE (Mean Absolute % Error)']
)

print("\n================ MODEL EVALUATION MATRIX ================")
print(comparison_table)
print("=========================================================")

# Step 4b: Visualize Performance Curves
plt.figure(figsize=(14, 7))
plt.plot(train_series.index[-200:], train_series.values[-200:], label="Historical Training Price (Truncated)", color='gray', alpha=0.6)
plt.plot(test_series.index, test_series.values, label="Actual Testing Price (Ground Truth)", color='black', linewidth=1.5)
plt.plot(test_series.index, arima_forecast, label="ARIMA Statistical Projection", color='#E53E3E', linestyle='--')
plt.plot(test_series.index, lstm_forecast, label="LSTM Neural Network Projection", color='#3182CE', linestyle=':')
plt.title("TSLA Out-of-Sample Forecast Evaluation: ARIMA vs LSTM", fontsize=14, fontweight='bold')
plt.xlabel("Timeline Date")
plt.ylabel("Adjusted Closing Price ($)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()